# E791 minimizer basin-of-attraction study

This notebook extends the single-toy diagnostic and quantifies how robustly the physical minimum is recovered as the starting point is moved away from the injected truth.

It intentionally uses the **same 50k-event toy** and the same likelihood as notebook 16. The goal is to choose a sensible start-randomization scale for the GenFit bias study without confusing fit bias with global-minimum stress testing.


In [ ]:
%run ./16_e791_single_toy_diagnostic.ipynb


## 1. Basin scan configuration

For each width $\sigma_{start}$, independent starts are generated as

$\theta_{start}=\theta_{true}+\mathcal{N}(0,\sigma_{start})$.

A fit is counted as recovering the physical minimum only if it is valid and its final NLL is compatible with the best minimum found by the truth-start and near-truth fits. `Minuit.valid` alone is not sufficient.


In [ ]:
START_SIGMAS = (0.05, 0.10, 0.25, 0.50, 1.00)
N_STARTS_PER_SIGMA = 20
NLL_TOLERANCE = 1e-3
BASIN_SEED = 20260831

reference_nll = min(result["nll"] for result in results[:2])
print(f"reference physical-minimum NLL = {reference_nll:.9f}")


## 2. Run repeated starts on the same toy


In [ ]:
basin_rng = np.random.default_rng(BASIN_SEED)
basin_records = []

for sigma_start in START_SIGMAS:
    print(f"\nstart sigma = {sigma_start:.2f}")
    for i in range(N_STARTS_PER_SIGMA):
        start = truth_vector + basin_rng.normal(
            0.0, sigma_start, size=truth_vector.size
        )
        fit_result = run_fit(f"sigma={sigma_start:.2f} #{i:02d}", start)
        delta_best = fit_result["nll"] - reference_nll
        success = (
            fit_result["valid"]
            and np.isfinite(fit_result["nll"])
            and delta_best <= NLL_TOLERANCE
        )
        basin_records.append({
            "sigma_start": sigma_start,
            "trial": i,
            "success": bool(success),
            "delta_best_nll": float(delta_best),
            "delta_truth_nll": float(fit_result["delta_nll"]),
            "distance_to_truth": float(fit_result["distance_to_truth"]),
            "max_abs_coefficient": float(fit_result["max_abs_coefficient"]),
            "edm": float(fit_result["edm"]),
            "valid": bool(fit_result["valid"]),
            "values": fit_result["values"],
        })

    group = [r for r in basin_records if r["sigma_start"] == sigma_start]
    n_success = sum(r["success"] for r in group)
    print(
        f"  physical minimum recovered: {n_success}/{N_STARTS_PER_SIGMA} "
        f"({100*n_success/N_STARTS_PER_SIGMA:.1f}%)"
    )


## 3. Recovery-rate summary


In [ ]:
basin_summary = []
for sigma_start in START_SIGMAS:
    group = [r for r in basin_records if r["sigma_start"] == sigma_start]
    successes = np.asarray([r["success"] for r in group], dtype=bool)
    deltas = np.asarray([r["delta_best_nll"] for r in group], dtype=float)
    basin_summary.append({
        "sigma_start": sigma_start,
        "n": len(group),
        "n_success": int(np.sum(successes)),
        "success_rate": float(np.mean(successes)),
        "median_delta_nll": float(np.median(deltas)),
        "worst_delta_nll": float(np.max(deltas)),
    })

print(
    f"{'sigma(start)':>12s} {'success':>10s} {'rate':>10s} "
    f"{'median DeltaNLL':>18s} {'worst DeltaNLL':>17s}"
)
for row in basin_summary:
    print(
        f"{row['sigma_start']:12.2f} "
        f"{row['n_success']:4d}/{row['n']:<4d} "
        f"{100*row['success_rate']:9.1f}% "
        f"{row['median_delta_nll']:18.6g} "
        f"{row['worst_delta_nll']:17.6g}"
    )


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.8))
ax.plot(
    [row["sigma_start"] for row in basin_summary],
    [100.0 * row["success_rate"] for row in basin_summary],
    marker="o",
)
ax.set_xlabel("Gaussian start width")
ax.set_ylabel("Physical-minimum recovery rate [%]")
ax.set_ylim(-5, 105)
ax.set_title("Basin of attraction of the physical minimum")
ax.grid(True, alpha=0.25)
fig.tight_layout()
plt.show()


## 4. NLL outcomes by start width

This plot reveals whether failed starts populate one or more distinct local-minimum families.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_rng = np.random.default_rng(BASIN_SEED + 1)
for sigma_start in START_SIGMAS:
    deltas = np.asarray([
        r["delta_best_nll"] for r in basin_records
        if r["sigma_start"] == sigma_start
    ])
    x = np.full(deltas.shape, sigma_start, dtype=float)
    jitter = plot_rng.normal(0.0, 0.008, size=deltas.size)
    ax.scatter(
        x + jitter, deltas, s=32, alpha=0.75,
        label=f"sigma={sigma_start:.2f}",
    )

ax.axhline(NLL_TOLERANCE, linestyle="--", label="success tolerance")
ax.set_xlabel("Gaussian start width")
ax.set_ylabel("NLL(fit) - NLL(best physical minimum)")
ax.set_title("Local-minimum outcomes versus starting displacement")
ax.legend(ncol=2)
fig.tight_layout()
plt.show()


## 5. Minuit-valid is not equivalent to physical-minimum recovery


In [ ]:
print(
    f"{'sigma(start)':>12s} {'Minuit valid':>14s} "
    f"{'physical minimum':>18s}"
)
for sigma_start in START_SIGMAS:
    group = [r for r in basin_records if r["sigma_start"] == sigma_start]
    n_valid = sum(r["valid"] for r in group)
    n_success = sum(r["success"] for r in group)
    print(
        f"{sigma_start:12.2f} "
        f"{n_valid:6d}/{len(group):<6d} "
        f"{n_success:8d}/{len(group):<8d}"
    )


## 6. Decision for the GenFit bias study

Choose the largest start width for which physical-minimum recovery remains high and stable. That width is appropriate for the **fit-bias** ensemble because it perturbs the initial conditions without intentionally turning every toy into a global-minimization stress test.

Global robustness from arbitrary starts should remain a separate multistart/minimizer-validation problem.
